# 나이대별 분리 모델 실험 (젊은층 vs 고령층)

연령대별 Local AUC 분석에서, 만18-34세/35-37세(전체의 62%)의 Local AUC가
0.70~0.71로 가장 낮게 나왔습니다. 표본수가 충분히 크기 때문에(16만 건),
이전 IVF/DI 분리(DI 6,291건뿐이라 표본부족으로 손해)와는 상황이 다릅니다.

**실험**: 만37세를 기준으로 두 그룹으로 나눠서 각자 따로 5-Fold 모델을
학습하고, 통합 모델(0.73998)과 비교합니다. 일단 하이퍼파라미터는 동일하게
(`best_params.json`) 써서, "분리 자체"의 효과만 순수하게 봅니다.

torch 없이 LightGBM만 사용합니다.

## 1. 라이브러리, 설정, best_params 로드

In [1]:
import os
import json
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
RANDOM_STATE = 1004
CACHE_DIR = "blend_cache"

with open("best_params.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
best_params = loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

print("불러온 best_params 키:", list(best_params.keys()))

불러온 best_params 키: ['n_estimators', 'learning_rate', 'num_leaves', 'max_depth', 'min_child_samples', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'min_split_gain']


## 2. 데이터 전처리 + 나이대 그룹 분리

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


# 나이 컬럼은 그룹 분리용으로 따로 보관 (모델 입력에서는 그대로 유지, 제거 안 함)
age_raw = train_raw["시술 당시 나이"].values
young_ages = ["만18-34세", "만35-37세"]
young_mask = np.isin(age_raw, young_ages)

print(f"젊은층(만18-37세): {young_mask.sum()}건 ({young_mask.mean()*100:.1f}%)")
print(f"고령층(만38세 이상+미상): {(~young_mask).sum()}건 ({(1-young_mask.mean())*100:.1f}%)")

df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

df_le = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col].astype(str))

y = df_le[TARGET_COL].values.astype(np.float32)
X_lgbm = df_le.drop(columns=[TARGET_COL])

print(f"\n전처리 완료: {X_lgbm.shape}")

젊은층(만18-37세): 160256건 (62.5%)
고령층(만38세 이상+미상): 96095건 (37.5%)

전처리 완료: (256351, 67)


## 3. 젊은층 전용 모델 (5-Fold)

In [3]:
X_young = X_lgbm.loc[young_mask].reset_index(drop=True)
y_young = y[young_mask]

skf_young = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_young_only = np.zeros(len(y_young))

for fold, (tr_idx, val_idx) in enumerate(skf_young.split(X_young, y_young), start=1):
    m = LGBMClassifier(**best_params, random_state=RANDOM_STATE, verbose=-1)
    m.fit(X_young.iloc[tr_idx], y_young[tr_idx])
    oof_young_only[val_idx] = m.predict_proba(X_young.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y_young[val_idx], oof_young_only[val_idx])
    print(f"Fold {fold}/{N_SPLITS}  젊은층 전용 AUC: {fold_auc:.5f}")

score_young_only = roc_auc_score(y_young, oof_young_only)
print(f"\n젊은층 전용 모델 Local AUC: {score_young_only:.5f}")
print(f"(참고) 통합모델의 젊은층 Local AUC는 0.7030/0.7010 수준이었음")

Fold 1/5  젊은층 전용 AUC: 0.70686
Fold 2/5  젊은층 전용 AUC: 0.70572
Fold 3/5  젊은층 전용 AUC: 0.70731
Fold 4/5  젊은층 전용 AUC: 0.70168
Fold 5/5  젊은층 전용 AUC: 0.70168

젊은층 전용 모델 Local AUC: 0.70462
(참고) 통합모델의 젊은층 Local AUC는 0.7030/0.7010 수준이었음


## 4. 고령층 전용 모델 (5-Fold)

In [4]:
X_old = X_lgbm.loc[~young_mask].reset_index(drop=True)
y_old = y[~young_mask]

skf_old = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_old_only = np.zeros(len(y_old))

for fold, (tr_idx, val_idx) in enumerate(skf_old.split(X_old, y_old), start=1):
    m = LGBMClassifier(**best_params, random_state=RANDOM_STATE, verbose=-1)
    m.fit(X_old.iloc[tr_idx], y_old[tr_idx])
    oof_old_only[val_idx] = m.predict_proba(X_old.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y_old[val_idx], oof_old_only[val_idx])
    print(f"Fold {fold}/{N_SPLITS}  고령층 전용 AUC: {fold_auc:.5f}")

score_old_only = roc_auc_score(y_old, oof_old_only)
print(f"\n고령층 전용 모델 Local AUC: {score_old_only:.5f}")

Fold 1/5  고령층 전용 AUC: 0.76028
Fold 2/5  고령층 전용 AUC: 0.74713
Fold 3/5  고령층 전용 AUC: 0.75491
Fold 4/5  고령층 전용 AUC: 0.75488
Fold 5/5  고령층 전용 AUC: 0.75478

고령층 전용 모델 Local AUC: 0.75432


## 5. 전체 결합 후 통합모델과 비교

In [5]:
# 두 서브모델 OOF를 원래 행 순서로 합치기
oof_split = np.zeros(len(y))
oof_split[young_mask] = oof_young_only
oof_split[~young_mask] = oof_old_only

score_split = roc_auc_score(y, oof_split)

# 비교 기준: 통합모델(진짜 튜닝 LightGBM)
oof_unified = np.load(f"{CACHE_DIR}/oof_lgbm_tuned.npy")
score_unified = roc_auc_score(y, oof_unified)

print(f"통합 모델 전체 AUC:        {score_unified:.5f}")
print(f"나이분리 모델 전체 AUC:    {score_split:.5f}")
print(f"개선: {score_split - score_unified:+.5f}")
print()

# 젊은층 구간만 따로: 분리모델이 그 구간 Local AUC를 진짜로 올렸는지 확인
unified_young_local = roc_auc_score(y[young_mask], oof_unified[young_mask])
print(f"[젊은층 구간만] 통합모델 Local AUC: {unified_young_local:.5f}")
print(f"[젊은층 구간만] 분리모델 Local AUC: {score_young_only:.5f}")
print(f"[젊은층 구간만] 개선: {score_young_only - unified_young_local:+.5f}")
print()

if score_split - score_unified > 0.001:
    print("=> 노이즈를 넘는 의미 있는 개선입니다! 채택을 검토해보세요.")
else:
    print("=> 노이즈 수준입니다. 나이대로 모델을 나누는 것 자체는 큰 효과가 없는 것으로 보입니다.")

통합 모델 전체 AUC:        0.73998
나이분리 모델 전체 AUC:    0.73922
개선: -0.00076

[젊은층 구간만] 통합모델 Local AUC: 0.70518
[젊은층 구간만] 분리모델 Local AUC: 0.70462
[젊은층 구간만] 개선: -0.00055

=> 노이즈 수준입니다. 나이대로 모델을 나누는 것 자체는 큰 효과가 없는 것으로 보입니다.


## 6. 결과 저장 (참고용)

In [6]:
os.makedirs(CACHE_DIR, exist_ok=True)
np.save(f"{CACHE_DIR}/oof_age_split.npy", oof_split)
print("저장 완료: blend_cache/oof_age_split.npy")

저장 완료: blend_cache/oof_age_split.npy
